### import packages

In [2]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [3]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### rag pipeline

In [4]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)


In [5]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [6]:
import openai
from langsmith import traceable, get_current_run_tree 
# for cost calculation
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(url = "http://localhost:6333") # on docker image it called qdrant as defined

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding

def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    retrieved_context_ids = []
    retrived_context = []
    similarity_scores = []
    retrived_context_ratings = []
    for point in response.points:
        retrieved_context_ids.append(point.payload['parent_asin'])
        retrived_context.append(point.payload['preprocessed_description'])
        similarity_scores.append(point.score)
        retrived_context_ratings.append(point.payload['average_rating'])
        
    return {"retrieved_context_ids": retrieved_context_ids,
            "retrived_context": retrived_context,
            "similarity_scores": similarity_scores,
            "retrived_context_ratings": retrived_context_ratings}

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrived_context'], context['retrived_context_ratings']):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    prompt = f"""
    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - Do not use markdown formatting.

    Here is the context:
    {preprocessed_context}
    Here is the question:
    {question}
    """
    return prompt


def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )
    # current_run = get_current_run_tree()
    # if current_run:
    #     current_run.metadata['usage_metadata'] = {
    #         "input_tokens": response.usage.prompt_tokens,
    #         "output_tokens": response.usage.completion_tokens,
    #         "total_tokens": response.usage.total_tokens
    #     }
    return response


def rag_pipeline(question, k=5):
    retrieved_context = retrieve_from_qdrant(question, k=k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrived_context']
    }
    return final_answer


In [7]:
output = rag_pipeline("do you have any earphones")

In [8]:
output

{'data_object': RAGGenerationResponse(answer='Yes—one option currently available is the Mini Invisible Bluetooth Earbud (pink, 1 earphone) with built-in mic and Bluetooth 4.1 (Model ID: B075V4T5X2).'),
 'answer': 'Yes—one option currently available is the Mini Invisible Bluetooth Earbud (pink, 1 earphone) with built-in mic and Bluetooth 4.1 (Model ID: B075V4T5X2).',
 'question': 'do you have any earphones',
 'retrieved_context_ids': ['B075V4T5X2',
  'B007136D32',
  'B08DR4K78X',
  'B07ZCLS3LL',
  'B07FZ3G7SC'],
 'retrieved_context': ['Mini Invisible Bluetooth Earbud,V4.1 Stereo Wireless Bluetooth Earphone with Built-in Mic,Sports Noise Cancelling in-Ear Headphone for iPhone Samsung and Other Android Phones (pink5) Mini Invisible Earbud ( Only 1 earphone): more comfortable and Lightweight when wearing. One multifunction-button design, comes with S/M/L ear-tips with anti-shedding features that secure fit different people; it’s hidden, suitable for a variety of scenes. Super Quality Sound

In [9]:
display(output['answer'])

'Yes—one option currently available is the Mini Invisible Bluetooth Earbud (pink, 1 earphone) with built-in mic and Bluetooth 4.1 (Model ID: B075V4T5X2).'

### rag pipeline with grounding context 

In [10]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question") # only id needed since we can query qdrant for the rest
    description: str = Field(description="Description of the item used to answer the question") # why need description? i want to show a summary to the user because we do not have in the vector db
    

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to the context")

In [23]:
import openai
from langsmith import traceable, get_current_run_tree 
# for cost calculation
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(url = "http://localhost:6333") # on docker image it called qdrant as defined

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding

def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    retrieved_context_ids = []
    retrived_context = []
    similarity_scores = []
    retrived_context_ratings = []
    for point in response.points:
        retrieved_context_ids.append(point.payload['parent_asin'])
        retrived_context.append(point.payload['preprocessed_description'])
        similarity_scores.append(point.score)
        retrived_context_ratings.append(point.payload['average_rating'])
        
    return {"retrieved_context_ids": retrieved_context_ids,
            "retrived_context": retrived_context,
            "similarity_scores": similarity_scores,
            "retrived_context_ratings": retrived_context_ratings}

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrived_context'], context['retrived_context_ratings']):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    prompt = f"""
    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - If you are describing multiple products, list them out as a list.

    Here is the context:
    {preprocessed_context}
    Here is the question:
    {question}
    """
    return prompt


def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )
    # current_run = get_current_run_tree()
    # if current_run:
    #     current_run.metadata['usage_metadata'] = {
    #         "input_tokens": response.usage.prompt_tokens,
    #         "output_tokens": response.usage.completion_tokens,
    #         "total_tokens": response.usage.total_tokens
    #     }
    return response


def rag_pipeline(question, k=5):
    retrieved_context = retrieve_from_qdrant(question, k=k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrived_context']
    }
    return final_answer


In [ ]:
output = rag_pipeline("do you have any earphones", k=10)

In [28]:
print(output['answer'])

Yes—we have earphones available, including:
- Mini Invisible Bluetooth Earbud (pink5) — lightweight in-ear Bluetooth earbud with built-in mic and noise-cancelling; comes with S/M/L ear tips (only 1 earphone).


In [31]:
output = rag_pipeline("do you have any earphones", k=50)

In [ ]:
print(output['answer']) #verified that my dataset only has 1

Yes—there are earphones available. 

- Mini Invisible Bluetooth Earbud (pink5) — lightweight, single-earphone option, Bluetooth 4.1 with built-in mic, up to ~4 hours playtime, comes with S/M/L ear tips.


In [35]:
output = rag_pipeline("do you have any sd card", k=50)
print(output['answer'])

Yes—available SD cards:

- AIRDISK 512GB Micro SD Card (microSDXC, UHS-I), up to 100MB/s, A2, U3, V30, Class 10, includes adapter; 2-year limited warranty.


In [33]:
output = rag_pipeline("do you have any camera", k=50)
print(output['answer'])

Yes—one camera-related item available is:
- Neewer NW-760 HD Monitor (7" IPS) for Sony/Canon/Nikon/Panasonic cameras (camera, tripod, and battery not included).
